In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib nao esta instalado; as tabelas serao geradas e os graficos serao pulados.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 80)
if plt is not None:
    plt.style.use("seaborn-v0_8-whitegrid")

## 1. Carregamento dos dados

In [ ]:
candidate_paths = [
    Path("../dataset/Unificada13052026_Limpa.csv"),
    Path("../dataset/Unificada13052026.csv"),
    Path("base_dados_unificada.xlsx"),
    Path("dataset/Unificada13052026_Limpa.csv"),
    Path("dataset/Unificada13052026.csv"),
    Path("coleta_dados_espectrais/base_dados_unificada.xlsx"),
    Path("EstresseHidricoFinal/dataset/Unificada13052026_Limpa.csv"),
    Path("EstresseHidricoFinal/dataset/Unificada13052026.csv"),
    Path("EstresseHidricoFinal/coleta_dados_espectrais/base_dados_unificada.xlsx"),
]

dataset_path = next((path for path in candidate_paths if path.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Nao encontrei a base em nenhum dos caminhos esperados.")

if dataset_path.suffix.lower() == ".csv":
    df = pd.read_csv(dataset_path, sep=";", decimal=",")
elif dataset_path.suffix.lower() in {".xls", ".xlsx"}:
    df = pd.read_excel(dataset_path)
else:
    raise ValueError(f"Extensao de arquivo nao suportada: {dataset_path.suffix}")

print(f"Arquivo carregado: {dataset_path}")
print(f"Dimensao bruta: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas".replace(",", "."))
df.head()

## 2. Separacao entre metadados e bandas

In [ ]:
def is_band_column(column_name):
    return str(column_name).strip().isdigit()

first_band_idx = next(
    (idx for idx, column in enumerate(df.columns) if is_band_column(column)),
    None,
)
if first_band_idx is None:
    raise ValueError("Nenhuma coluna numerada foi encontrada para representar bandas.")

metadata_cols = list(df.columns[:first_band_idx])
band_cols = [column for column in df.columns[first_band_idx:] if is_band_column(column)]
non_band_after_first = [column for column in df.columns[first_band_idx:] if not is_band_column(column)]

band_wavelengths = np.array([int(str(column).strip()) for column in band_cols])
bands = df[band_cols].apply(pd.to_numeric, errors="coerce")

print(f"Colunas de metadados ({len(metadata_cols)}): {metadata_cols}")
print(f"Bandas numeradas ({len(band_cols)}): {band_wavelengths.min()} a {band_wavelengths.max()} nm")
print(f"Passo mediano entre bandas: {np.median(np.diff(np.sort(band_wavelengths))):.0f} nm")

if non_band_after_first:
    print("Atencao: existem colunas nao numeradas apos a primeira banda:")
    print(non_band_after_first)

## 3. Resumo geral do dataset

In [ ]:
band_diffs = np.diff(np.sort(band_wavelengths))
resumo_geral = pd.DataFrame(
    {
        "metrica": [
            "amostras",
            "colunas totais",
            "colunas de metadados",
            "bandas hiperspectrais",
            "primeira banda (nm)",
            "ultima banda (nm)",
            "passo predominante (nm)",
            "valores ausentes totais",
            "linhas duplicadas",
            "menor reflectancia",
            "maior reflectancia",
            "reflectancia media global",
        ],
        "valor": [
            len(df),
            df.shape[1],
            len(metadata_cols),
            len(band_cols),
            band_wavelengths.min(),
            band_wavelengths.max(),
            pd.Series(band_diffs).mode().iloc[0] if len(band_diffs) else np.nan,
            int(df.isna().sum().sum()),
            int(df.duplicated().sum()),
            bands.min().min(),
            bands.max().max(),
            bands.stack().mean(),
        ],
    }
)
resumo_geral

## 4. Caracterizacao dos metadados

In [ ]:
metadata_profile = []
for column in metadata_cols:
    values = df[column]
    top_values = values.value_counts(dropna=False).head(5)
    metadata_profile.append(
        {
            "coluna": column,
            "tipo": str(values.dtype),
            "valores_unicos": values.nunique(dropna=True),
            "ausentes": int(values.isna().sum()),
            "mais_frequentes": "; ".join(f"{idx}: {count}" for idx, count in top_values.items()),
        }
    )

pd.DataFrame(metadata_profile)

In [ ]:
for column in metadata_cols:
    cardinality = df[column].nunique(dropna=False)
    if cardinality <= 30:
        display(
            df[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="amostras")
        )

## 5. Quantidade de amostras por grupos experimentais

In [ ]:
preferred_groupings = [
    ["condicao"],
    ["genotipo"],
    ["bloco"],
    ["data_coleta"],
    ["turno"],
    ["condicao", "genotipo"],
    ["condicao", "data_coleta"],
    ["genotipo", "condicao", "bloco"],
]

for grouping in preferred_groupings:
    available_grouping = [column for column in grouping if column in df.columns]
    if len(available_grouping) != len(grouping):
        continue

    print("Amostras por " + " + ".join(available_grouping))
    display(
        df.groupby(available_grouping, dropna=False)
        .size()
        .reset_index(name="amostras")
        .sort_values(available_grouping)
    )

## 6. Qualidade e consistencia dos dados

In [ ]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
missing_by_column = missing_by_column[missing_by_column > 0]

zero_variance_bands = bands.columns[bands.nunique(dropna=True) <= 1].tolist()
unexpected_band_steps = pd.Series(band_diffs).value_counts().sort_index().rename_axis("passo_nm").reset_index(name="ocorrencias")

quality_summary = pd.DataFrame(
    {
        "checagem": [
            "colunas com valores ausentes",
            "bandas com variancia zero",
            "passos distintos entre bandas",
            "linhas duplicadas",
        ],
        "resultado": [
            len(missing_by_column),
            len(zero_variance_bands),
            len(unexpected_band_steps),
            int(df.duplicated().sum()),
        ],
    }
)

display(quality_summary)
display(unexpected_band_steps)

if len(missing_by_column):
    display(missing_by_column.reset_index(name="ausentes").rename(columns={"index": "coluna"}))
if zero_variance_bands:
    print("Bandas com variancia zero:", zero_variance_bands[:20])

## 7. Estatisticas das bandas

In [ ]:
band_stats = pd.DataFrame(
    {
        "comprimento_onda_nm": band_wavelengths,
        "media": bands.mean().to_numpy(),
        "desvio_padrao": bands.std().to_numpy(),
        "minimo": bands.min().to_numpy(),
        "p25": bands.quantile(0.25).to_numpy(),
        "mediana": bands.median().to_numpy(),
        "p75": bands.quantile(0.75).to_numpy(),
        "maximo": bands.max().to_numpy(),
    }
)

display(band_stats.head(10))
display(band_stats.tail(10))

In [ ]:
if plt is None:
    print("Grafico pulado: instale matplotlib para visualizar o perfil espectral medio.")
else:
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(band_stats["comprimento_onda_nm"], band_stats["media"], color="#245f73", label="media")
    ax.fill_between(
        band_stats["comprimento_onda_nm"],
        band_stats["media"] - band_stats["desvio_padrao"],
        band_stats["media"] + band_stats["desvio_padrao"],
        color="#9abf88",
        alpha=0.25,
        label="media +/- desvio padrao",
    )
    ax.set_title("Perfil espectral medio")
    ax.set_xlabel("Comprimento de onda (nm)")
    ax.set_ylabel("Reflectancia")
    ax.legend()
    plt.show()

In [ ]:
if plt is None:
    print("Grafico pulado: instale matplotlib para visualizar os espectros amostrados.")
else:
    sample_size = min(25, len(df))
    sampled_rows = df.sample(sample_size, random_state=42).index

    fig, ax = plt.subplots(figsize=(13, 5))
    for row_idx in sampled_rows:
        ax.plot(band_wavelengths, bands.loc[row_idx].to_numpy(), alpha=0.35, linewidth=1)

    ax.set_title(f"{sample_size} espectros amostrados aleatoriamente")
    ax.set_xlabel("Comprimento de onda (nm)")
    ax.set_ylabel("Reflectancia")
    plt.show()

## 8. Comparacao espectral por grupos

In [ ]:
def plot_mean_spectra_by_group(group_col, max_groups=12):
    if plt is None:
        print(f"Grafico por {group_col} pulado: instale matplotlib para visualizar.")
        return

    if group_col not in df.columns:
        print(f"Coluna ausente: {group_col}")
        return

    counts = df[group_col].value_counts(dropna=False)
    groups_to_plot = counts.head(max_groups).index

    fig, ax = plt.subplots(figsize=(13, 5))
    for group_value in groups_to_plot:
        mask = df[group_col].eq(group_value)
        ax.plot(
            band_wavelengths,
            bands.loc[mask].mean().to_numpy(),
            linewidth=2,
            label=f"{group_value} (n={mask.sum()})",
        )

    ax.set_title(f"Espectro medio por {group_col}")
    ax.set_xlabel("Comprimento de onda (nm)")
    ax.set_ylabel("Reflectancia media")
    ax.legend(ncol=2, fontsize=9)
    plt.show()

for group_col in ["condicao", "genotipo", "turno", "data_coleta"]:
    plot_mean_spectra_by_group(group_col)

In [ ]:
if plt is None:
    print("Mapa de calor pulado: instale matplotlib para visualizar.")
elif {"condicao", "genotipo"}.issubset(df.columns):
    group_labels = []
    group_means = []

    for (condicao, genotipo), idx in df.groupby(["condicao", "genotipo"], dropna=False).groups.items():
        group_labels.append(f"{condicao} | {genotipo} (n={len(idx)})")
        group_means.append(bands.loc[idx].mean().to_numpy())

    group_means = np.vstack(group_means)

    fig, ax = plt.subplots(figsize=(14, 4 + 0.35 * len(group_labels)))
    image = ax.imshow(group_means, aspect="auto", cmap="viridis")
    ax.set_title("Mapa de calor das reflectancias medias por condicao e genotipo")
    ax.set_xlabel("Comprimento de onda (nm)")
    ax.set_yticks(np.arange(len(group_labels)))
    ax.set_yticklabels(group_labels)

    tick_positions = np.linspace(0, len(band_wavelengths) - 1, 9, dtype=int)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(band_wavelengths[tick_positions])
    fig.colorbar(image, ax=ax, label="Reflectancia media")
    plt.show()

## 9. Principais leituras automaticas

In [ ]:
print("Resumo interpretativo")
print(f"- A base possui {len(df)} amostras e {len(band_cols)} bandas hiperspectrais.")
print(f"- As bandas cobrem de {band_wavelengths.min()} a {band_wavelengths.max()} nm.")
print(f"- Foram identificadas {len(metadata_cols)} colunas de metadados: {', '.join(map(str, metadata_cols))}.")
print(f"- Valores ausentes totais: {int(df.isna().sum().sum())}.")
print(f"- Linhas duplicadas: {int(df.duplicated().sum())}.")

def collection_date_key(value):
    value = str(value)
    number = "".join(char for char in value if char.isdigit())
    suffix = "".join(char for char in value if char.isalpha())
    return (int(number) if number else 9999, suffix)

def sort_counts_by_collection_date(counts):
    ordered_index = sorted(counts.index, key=collection_date_key)
    return counts.reindex(ordered_index)

for column in ["condicao", "genotipo", "bloco", "data_coleta", "turno"]:
    if column in df.columns:
        counts = df[column].value_counts(dropna=False)
        if column == "data_coleta":
            counts = sort_counts_by_collection_date(counts)
        formatted = "; ".join(f"{idx}: {value}" for idx, value in counts.items())
        print(f"- Distribuicao por {column}: {formatted}.")